In [1]:
import uuid
from typing import List, Dict 
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langgraph.prebuilt import create_react_agent

## Prepare Models

In [2]:
load_dotenv()

True

In [3]:
llm = ChatOpenAI(
    model="gpt-5.4-mini",
    temperature=0
)
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

## Example Raw Docs

In [4]:
raw_texts = [
    """
    LangGraph is a framework for building stateful multi-agent systems.
    It extends traditional LLM application development by introducing
    explicit workflow graphs composed of nodes and edges. Each node
    performs a specific task, such as calling an LLM, executing a tool,
    retrieving documents, or performing custom business logic.

    One of LangGraph's most important features is its shared state.
    Instead of passing variables manually between functions, every node
    can read and update a centralized state object. This makes it easier
    to build long-running workflows, conversational agents, and systems
    that require memory across multiple reasoning steps.

    LangGraph also supports conditional routing through conditional
    edges. Rather than following a fixed sequence of operations, the
    workflow can dynamically choose the next node depending on the
    current state or the LLM's decision. This enables branching,
    retries, fallback logic, human approval steps, and error recovery.

    Tool execution is another important capability. An LLM can decide
    whether to call external tools such as web search, SQL databases,
    APIs, calculators, or Python functions. The tool results are then
    stored in the shared state and used during later reasoning steps.

    LangGraph is commonly used for AI agents, retrieval pipelines,
    autonomous workflows, planning systems, and production-grade
    multi-agent architectures where maintaining state and controlling
    execution flow are important.
    """,

    """
    Retrieval-Augmented Generation, commonly called RAG, combines
    information retrieval with large language models. Instead of relying
    only on the model's internal knowledge, RAG first retrieves relevant
    documents from an external knowledge base and supplies those
    documents as additional context during generation.

    A typical RAG pipeline begins by loading documents from PDFs,
    websites, databases, or other sources. These documents are split
    into smaller chunks before being converted into vector embeddings
    using an embedding model. The embeddings are stored inside a vector
    database such as FAISS, Chroma, Pinecone, or Milvus.

    When a user submits a query, the same embedding model converts the
    query into a vector representation. A similarity search identifies
    the most relevant document chunks, which are then passed into the
    language model as context. This allows the model to answer questions
    using current or private information that was unavailable during
    pretraining.

    Advanced RAG systems often include metadata filtering, hybrid search,
    re-ranking models, parent-child retrieval, contextual compression,
    and caching to improve both retrieval accuracy and response speed.
    RAG is widely used for enterprise chatbots, document assistants,
    question-answering systems, and knowledge management applications.
    """,

    """
    LoRA, which stands for Low-Rank Adaptation, is a parameter-efficient
    fine-tuning technique for large language models. Instead of updating
    every weight inside the neural network, LoRA freezes the original
    model parameters and trains only a small number of additional
    low-rank matrices.

    Because only a tiny fraction of the parameters are updated, LoRA
    dramatically reduces GPU memory usage, storage requirements, and
    training time. This allows developers to fine-tune models with
    relatively modest hardware while still achieving competitive
    performance on specialized tasks.

    After training, the adapter weights can either remain separate from
    the base model or be merged into the original model for deployment.
    Multiple LoRA adapters can also be created for different domains,
    allowing a single base model to support many specialized tasks
    without storing multiple complete copies of the model.

    LoRA is commonly used for domain adaptation, instruction tuning,
    chatbot customization, code generation, and document understanding.
    Compared with full fine-tuning, it is significantly more efficient
    while preserving most of the original model's capabilities. This
    makes LoRA one of the most widely adopted techniques for adapting
    modern large language models.
    """
]

In [5]:
documents = [
    Document(page_content=text, metadata={"doc_id": str(i)})
    for i, text in enumerate(raw_texts)
]

In [6]:
documents[0]

Document(metadata={'doc_id': '0'}, page_content="\n    LangGraph is a framework for building stateful multi-agent systems.\n    It extends traditional LLM application development by introducing\n    explicit workflow graphs composed of nodes and edges. Each node\n    performs a specific task, such as calling an LLM, executing a tool,\n    retrieving documents, or performing custom business logic.\n\n    One of LangGraph's most important features is its shared state.\n    Instead of passing variables manually between functions, every node\n    can read and update a centralized state object. This makes it easier\n    to build long-running workflows, conversational agents, and systems\n    that require memory across multiple reasoning steps.\n\n    LangGraph also supports conditional routing through conditional\n    edges. Rather than following a fixed sequence of operations, the\n    workflow can dynamically choose the next node depending on the\n    current state or the LLM's decision. 

## Function for Generating summaries 

In [7]:
def generate_summary(doc):
    """
    Generate a summary for each document 
    """
    prompt = f"""
    Generate a summary from the given text below:

    Provided Text:
    {doc}

    Rules: 
    1. Only return the summary without any addtional comments or extra words
    2. Text limit is within 300 words
    """.strip()
    
    return llm.invoke(prompt).content
    

In [8]:
generate_summary(raw_texts[0])

'LangGraph is a framework for building stateful multi-agent systems using explicit workflow graphs made of nodes and edges. Each node performs a task such as calling an LLM, using tools, retrieving documents, or running custom logic. Its shared state allows nodes to read and update a centralized memory, making it well suited for long-running workflows, conversational agents, and multi-step reasoning. Conditional edges enable dynamic routing based on state or LLM decisions, supporting branching, retries, fallback paths, human approval, and error recovery. LangGraph also integrates tool execution, allowing LLMs to call external resources like search engines, databases, APIs, calculators, or Python functions and store the results in shared state. It is commonly used for AI agents, retrieval pipelines, autonomous workflows, planning systems, and production-grade multi-agent architectures.'

## Split documents into detailed chunks


In [9]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

all_summaries = []
all_chunks = []

for doc in documents:
    
    parent_id = str(uuid.uuid4())

    # for summary vectors
    summary_data = Document(
        page_content= generate_summary(doc),
        metadata={
            "parent_id": parent_id,
            "source_doc_id": doc.metadata["doc_id"]
        }
    )

    # for chunk vectors
    chunks = splitter.split_documents([doc])
    print("CHUNKS", chunks)
    for chunk in chunks: 
        chunk.metadata["parent_id"] = parent_id
        chunk.metadata["source_doc_id"] = doc.metadata["doc_id"]

    all_summaries.append(summary_data)
    all_chunks.extend(chunks)

CHUNKS [Document(metadata={'doc_id': '0'}, page_content='LangGraph is a framework for building stateful multi-agent systems.\n    It extends traditional LLM application development by introducing\n    explicit workflow graphs composed of nodes and edges. Each node\n    performs a specific task, such as calling an LLM, executing a tool,'), Document(metadata={'doc_id': '0'}, page_content='retrieving documents, or performing custom business logic.'), Document(metadata={'doc_id': '0'}, page_content="One of LangGraph's most important features is its shared state.\n    Instead of passing variables manually between functions, every node\n    can read and update a centralized state object. This makes it easier\n    to build long-running workflows, conversational agents, and systems"), Document(metadata={'doc_id': '0'}, page_content='that require memory across multiple reasoning steps.'), Document(metadata={'doc_id': '0'}, page_content="LangGraph also supports conditional routing through condit

In [10]:
all_chunks

[Document(metadata={'doc_id': '0', 'parent_id': '25c9d33c-168a-4970-b41f-b309b16320d5', 'source_doc_id': '0'}, page_content='LangGraph is a framework for building stateful multi-agent systems.\n    It extends traditional LLM application development by introducing\n    explicit workflow graphs composed of nodes and edges. Each node\n    performs a specific task, such as calling an LLM, executing a tool,'),
 Document(metadata={'doc_id': '0', 'parent_id': '25c9d33c-168a-4970-b41f-b309b16320d5', 'source_doc_id': '0'}, page_content='retrieving documents, or performing custom business logic.'),
 Document(metadata={'doc_id': '0', 'parent_id': '25c9d33c-168a-4970-b41f-b309b16320d5', 'source_doc_id': '0'}, page_content="One of LangGraph's most important features is its shared state.\n    Instead of passing variables manually between functions, every node\n    can read and update a centralized state object. This makes it easier\n    to build long-running workflows, conversational agents, and sys

## Register summaries & chunks
Just creating in-memory vectorstores 

In [11]:
summary_vectorstore = Chroma(
    collection_name="summary_index",
    embedding_function=embedding_model
)

summary_vectorstore.add_documents(all_summaries)


chunk_vectorstore = Chroma(
    collection_name="chunk_index",
    embedding_function=embedding_model
)

chunk_vectorstore.add_documents(all_chunks)

['ff5b93cd-1073-4924-bb12-d9ba42e58a2c',
 '7e33a0a7-6b9f-49f9-b78e-e94d2f53302f',
 'fd572c87-13cf-4053-a556-2b7bee2c4291',
 '89a230c5-6e38-489f-9fc4-8e1c47e0d0a6',
 '2368a34f-f4e3-42a5-a554-cce5414f623b',
 '7d15713b-224b-4076-8138-9ab01dd0828d',
 'd55cdaf4-61af-48a0-b813-3cc41f052603',
 '10b8097c-de4f-4dfe-b27a-67f1ffd5c8c9',
 'dc3eb4bd-2dd7-4eaa-9214-6e4d3d7c74e2',
 '073db743-e164-4662-b401-8bf58a54b1c1',
 'b0cbc574-bc20-4cc9-909f-e47667b953ba',
 '56b6fe64-8086-4f24-b3fd-a24019046d42',
 '72316da7-b1c9-43ad-a870-9bbbc2f2d017',
 '060d542f-823e-4387-be88-22aecce7ef83',
 '44242f62-7a8b-4c14-bcea-e4adc4dc7bf5',
 '242c018d-9a4d-403a-b299-87369a717c2e',
 '6574f5f9-092d-419b-bcd2-9564b3a8cad0',
 '08bd36e8-fd58-4f56-a5b5-f6863576fa70',
 '78b0ae95-52a0-45ad-986a-1f29a324d7ab',
 'd8bf6ea5-18cc-4603-9061-d8d24bce7eb2',
 'ff09736d-f793-4eb4-96e1-0f8a7fa89247',
 'e5d6d0e1-b81f-4de7-a4e3-897c3c5c6ae6',
 '202ed1b1-83c1-4858-8696-0e8c4ba4ed50',
 '5f6e1df9-8faa-4ef3-b13a-16e28cbcb25b']

## Retrieve 

In [12]:
def hierarchical_retrieve(
    query: str,
    summary_k= 1, 
    chunk_k= 3,
):
    summary_results = summary_vectorstore.similarity_search(query=query, k=summary_k)
    summary_parent_ids = [
        summary_result.metadata.get("parent_id")
        for summary_result in summary_results
    ]

    retrieved_chunks = []
    for parent_id in summary_parent_ids:
        chunks = chunk_vectorstore.similarity_search(
            query=query,
            k=chunk_k,
            filter={"parent_id": parent_id}
        )

        retrieved_chunks.extend(chunks)

    return retrieved_chunks


## Inject Context to LLM

In [13]:
def build_context(docs: List[Document]) -> str:
    """
    Return a string that contains retrieved document content.

    Args:
        docs: documents retrieved from RAG pipeline. 
    """

    context_parts = []

    for doc in docs:
        doc_id = doc.metadata.get("source_doc_id", "unknown")
        chunk_id = doc.id
        text = doc.page_content

        context_parts.append(
            f"""
            [source_doc_id: {doc_id} | chunk_id: {chunk_id}]
            {text}
            """.strip()
        )
    
    context = "\n\n".join(context_parts)
    
    return context

## Retriever tool

In [14]:
@tool
def search_knowledge_base(query: str) -> str:
    """
    Search the private RAG knowledge base when the user asks about
    LangGraph, RAG, LoRA, retrieval, fine-tuning, agents, vector databases,
    or related technical concepts.
    """

    docs = hierarchical_retrieve(query)
    context = build_context(docs)

    return context

## Create Agent

In [24]:
system_message = """
You are a helpful RAG assiatant that answers based on the retrieved context. 

When you use retrieved context:
- Answer based on the retrieved context.
- Cite source_doc_id and chunk_id.
- If the retrieved context does not contain the answer, say you do not know.

Return in this format:
{Your Answer}

{Doc_ID} | {Chunk_ID}
"""

agent = create_react_agent(
    model=llm,
    tools=[search_knowledge_base],
    prompt=system_message
)

/var/folders/y7/jzgx63gj4rb9p1g3z163h7l80000gn/T/ipykernel_5023/707118067.py:15: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


## Chat Function

In [25]:
def chat(user_input: str) -> str: 
    result = agent.invoke(
        {
            "messages": [
                HumanMessage(content=user_input)
            ]
        }
    )

    return result

In [ ]:
if __name__ == "__main__": 
    initial_msg = "Hi there! What do I help you with today?"
    print(f"Assistant: {initial_msg}")
    
    while True:
        user_input = input("\nUser: ")

        if user_input.lower() in ["quit", "exit"]:
            break

        answer = chat(user_input)

        print(f"Assistant: {answer.get("messages")[-1].content}")

Assistant: Hi there! What do I help you with today?
